In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Safe Colab install cell
# IMPORTANT: do not use -U/--upgrade and do not install torch/torchaudio/cuda here.
# Colab already provides compatible torch, torchvision, torchaudio, pandas, and CUDA packages.
!pip -q install "jedi>=0.16"
!pip -q install "funasr==1.4.1" "modelscope==1.39.1" "librosa==0.11.0" "soundfile" "tqdm" "scikit-learn" --upgrade-strategy only-if-needed

import torch, pandas as pd, numpy as np
print("torch:", torch.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 91.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 28.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 976.3/976.3 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 146.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.0/156.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.1 MB/s eta 0:00:00


In [3]:
import os
import json
import random
import warnings
import hashlib
import platform
import datetime
import importlib.metadata as im
import numpy as np
import pandas as pd
import librosa
import soundfile as sf

from pathlib import Path
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PROJECT = Path("/content/drive/MyDrive/New Jurnal Cross")

CSV_DIR = BASE_PROJECT / "processed_intra_csv"
OUT_DIR = BASE_PROJECT / "processed_intra_sequence_features_plus_base"
TMP_AUDIO_DIR = BASE_PROJECT / "tmp_e2v_seq_audio_plus_base"

OUT_DIR.mkdir(parents=True, exist_ok=True)
TMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["emodb", "ravdess", "resd"]

CSV_FILES = {
    "emodb": CSV_DIR / "split_emodb_6class_optimized.csv",
    "ravdess": CSV_DIR / "split_ravdess_6class_optimized.csv",
    "resd": CSV_DIR / "split_resd_6class_optimized.csv",
}

LABELS = ["angry", "disgust", "fear", "happy", "neutral", "sad"]

LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

CONFIG = {
    "sample_rate": 16000,
    "duration": 4.0,

    # handcrafted frame-level
    "n_mfcc": 40,
    "hop_length": 320,   # 20 ms at 16 kHz
    "n_fft": 1024,

    # sequence control
    "target_frames": 200,  # 4 seconds / 20 ms = approx 200 frames

    # emotion2vec
    "model_name": "iic/emotion2vec_plus_base",
    "granularity": "frame",
    "extract_embedding": True,
    "expected_e2v_dim": 768,

    "labels": LABELS,
}

print("OUT_DIR:", OUT_DIR)
for ds, p in CSV_FILES.items():
    print(ds, p.exists(), p)

OUT_DIR: /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base
emodb True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_emodb_6class_optimized.csv
ravdess True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_ravdess_6class_optimized.csv
resd True /content/drive/MyDrive/New Jurnal Cross/processed_intra_csv/split_resd_6class_optimized.csv


In [4]:
from funasr import AutoModel

e2v_model = AutoModel(
    model=CONFIG["model_name"],
    trust_remote_code=True,
    device="cuda:0",      # ganti "cpu" jika tidak pakai GPU
    disable_update=True   # penting untuk reproducibility
)

print("Loaded:", CONFIG["model_name"])

funasr version: 1.4.1.


2026-08-08 14:13:46,591 | INFO    | modelscope_hub.download | Downloading 11 files from iic/emotion2vec_plus_base@master


Downloading:   0%|          | 0/11 [00:00<?, ?file/s]

config.yaml:   0%|          | 0.00/3.17k [00:00<?, ?B/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

...

model.pt:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

test.wav:   0%|          | 0.00/131k [00:00<?, ?B/s]

tokens.txt:   0%|          | 0.00/120 [00:00<?, ?B/s]

Loading remote code failed: model, No module named 'model'
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.weight, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.0.0.bias, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.weight, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.1.0.bias, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.weight, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Warning, miss key in ckpt: modality_encoders.AUDIO.decoder.blocks.2.0.bias, /root/.cache/modelscope/models/iic--emotion2vec_plus_base/s

In [5]:
def pkg_ver(package_name):
    try:
        return im.version(package_name)
    except Exception:
        return "not installed"


def sha256_of_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def find_modelscope_checkpoint(model_id):
    """
    Cari checkpoint ModelScope/FunASR secara robust.
    Untuk iic/emotion2vec_plus_base, biasanya file checkpoint adalah model.pt.
    """
    org, name = model_id.split("/", 1)

    candidate_roots = [
        Path.home() / ".cache" / "modelscope" / "hub" / "models" / org / name,
        Path("/root/.cache/modelscope/hub/models") / org / name,
        Path.home() / ".cache" / "modelscope" / "hub" / org / name,
        Path("/root/.cache/modelscope/hub") / org / name,
        Path.home() / ".cache" / "modelscope" / "models" / f"{org}--{name}",
        Path("/root/.cache/modelscope/models") / f"{org}--{name}",
    ]

    exts = [".pt", ".pth", ".bin", ".safetensors"]
    candidates = []

    for root in candidate_roots:
        if root.exists():
            for ext in exts:
                candidates.extend(root.rglob(f"*{ext}"))

    candidates = sorted(
        list({p.resolve() for p in candidates if p.is_file()}),
        key=lambda p: p.stat().st_size,
        reverse=True
    )

    if len(candidates) == 0:
        return None, []

    return candidates[0], candidates


checkpoint_path, checkpoint_candidates = find_modelscope_checkpoint(CONFIG["model_name"])

if checkpoint_path is not None:
    checkpoint_sha256 = sha256_of_file(checkpoint_path)
    checkpoint_size_mb = checkpoint_path.stat().st_size / (1024 ** 2)
else:
    checkpoint_sha256 = None
    checkpoint_size_mb = None

sequence_environment_manifest = {
    "notebook": "08b_extract_sequence_features_intra.ipynb",
    "task": "emotion2vec frame-level sequence feature extraction plus handcrafted sequence extraction",
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "base_project": str(BASE_PROJECT),
    "csv_dir": str(CSV_DIR),
    "out_dir": str(OUT_DIR),
    "tmp_audio_dir": str(TMP_AUDIO_DIR),

    "model_id": CONFIG["model_name"],
    "granularity": CONFIG["granularity"],
    "extract_embedding": CONFIG["extract_embedding"],
    "expected_e2v_dim": CONFIG["expected_e2v_dim"],
    "target_frames": CONFIG["target_frames"],
    "hop_length": CONFIG["hop_length"],
    "sample_rate": CONFIG["sample_rate"],
    "duration": CONFIG["duration"],

    "checkpoint_path": str(checkpoint_path) if checkpoint_path is not None else None,
    "checkpoint_size_mb": round(checkpoint_size_mb, 2) if checkpoint_size_mb is not None else None,
    "checkpoint_sha256": checkpoint_sha256,
    "checkpoint_candidates": [str(p) for p in checkpoint_candidates],

    "python_version": platform.python_version(),
    "funasr_version": pkg_ver("funasr"),
    "modelscope_version": pkg_ver("modelscope"),
    "torch_version": pkg_ver("torch"),
    "torchaudio_version": pkg_ver("torchaudio"),
    "numpy_version": pkg_ver("numpy"),
    "pandas_version": pkg_ver("pandas"),
    "librosa_version": pkg_ver("librosa"),
    "soundfile_version": pkg_ver("soundfile"),
    "scikit_learn_version": pkg_ver("scikit-learn"),
}

with open(OUT_DIR / "sequence_environment_manifest.json", "w") as f:
    json.dump(sequence_environment_manifest, f, indent=2)

print("=" * 80)
print("SEQUENCE FEATURE EXTRACTION REPRODUCIBILITY MANIFEST")
print("=" * 80)
print("Model ID          :", sequence_environment_manifest["model_id"])
print("Checkpoint path   :", sequence_environment_manifest["checkpoint_path"])
print("Checkpoint SHA256 :", sequence_environment_manifest["checkpoint_sha256"])
print("Checkpoint size   :", sequence_environment_manifest["checkpoint_size_mb"], "MB")
print("FunASR version    :", sequence_environment_manifest["funasr_version"])
print("ModelScope ver.   :", sequence_environment_manifest["modelscope_version"])
print("Torch version     :", sequence_environment_manifest["torch_version"])
print("Saved manifest    :", OUT_DIR / "sequence_environment_manifest.json")

SEQUENCE FEATURE EXTRACTION REPRODUCIBILITY MANIFEST
Model ID          : iic/emotion2vec_plus_base
Checkpoint path   : /root/.cache/modelscope/models/iic--emotion2vec_plus_base/snapshots/master/model.pt
Checkpoint SHA256 : 60710b5aae1dbe69bdac8920028fb05882d4314fd09031922b4b61ee9e7aadbd
Checkpoint size   : 1066.44 MB
FunASR version    : 1.4.1
ModelScope ver.   : 1.39.1
Torch version     : 2.11.0+cu128
Saved manifest    : /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base/sequence_environment_manifest.json


In [6]:
def load_audio_fixed(path, sr=16000, duration=4.0):
    audio, _ = librosa.load(path, sr=sr, mono=True)

    target_len = int(sr * duration)

    if len(audio) >= target_len:
        start = (len(audio) - target_len) // 2
        audio = audio[start:start + target_len]
    else:
        pad_len = target_len - len(audio)
        if len(audio) > 1:
            audio = np.pad(audio, (0, pad_len), mode="reflect")
        else:
            audio = np.pad(audio, (0, pad_len), mode="constant")

    return audio.astype(np.float32)

In [7]:
def cmvn_per_utterance(mfcc):
    mean = mfcc.mean(axis=1, keepdims=True)
    std = mfcc.std(axis=1, keepdims=True) + 1e-8
    return (mfcc - mean) / std


def extract_handcrafted_sequence(audio, cfg):
    """
    Output shape: [T, 43]
    Components per frame:
      MFCC 40
      RMS 1
      ZCR 1
      F0 1
    """
    sr = cfg["sample_rate"]
    hop = cfg["hop_length"]
    n_fft = cfg["n_fft"]

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=cfg["n_mfcc"],
        hop_length=hop,
        n_fft=n_fft
    )
    mfcc = cmvn_per_utterance(mfcc)  # [40, T]

    rms = librosa.feature.rms(
        y=audio,
        hop_length=hop,
        frame_length=n_fft
    )  # [1, T]

    zcr = librosa.feature.zero_crossing_rate(
        y=audio,
        hop_length=hop,
        frame_length=n_fft
    )  # [1, T]

    try:
        f0 = librosa.yin(
            audio,
            fmin=50,
            fmax=500,
            sr=sr,
            hop_length=hop,
            frame_length=n_fft
        )
        f0 = np.nan_to_num(f0, nan=0.0, posinf=0.0, neginf=0.0)

        # librosa.yin often returns fmin for unvoiced-ish frames
        f0 = np.where(f0 <= 51, 0.0, f0)
        f0 = f0.reshape(1, -1)
    except Exception:
        f0 = np.zeros((1, mfcc.shape[1]), dtype=np.float32)

    # Samakan panjang frame
    min_t = min(mfcc.shape[1], rms.shape[1], zcr.shape[1], f0.shape[1])

    mfcc = mfcc[:, :min_t]
    rms = rms[:, :min_t]
    zcr = zcr[:, :min_t]
    f0 = f0[:, :min_t]

    feat = np.concatenate([mfcc, rms, zcr, f0], axis=0)  # [43, T]
    feat = feat.T.astype(np.float32)                    # [T, 43]
    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)

    return feat

In [8]:
def parse_e2v_frame_output(res):
    """
    Ambil frame-level embedding dari output FunASR.
    Expected output ideally: [T, D]
    """
    item = res[0] if isinstance(res, list) else res

    possible_keys = [
        "feats",
        "feature",
        "embedding",
        "embeddings",
    ]

    for key in possible_keys:
        if isinstance(item, dict) and key in item:
            arr = np.asarray(item[key], dtype=np.float32)
            return arr

    if isinstance(item, dict):
        candidates = []
        for k, v in item.items():
            try:
                arr = np.asarray(v, dtype=np.float32)
                if arr.ndim >= 2 and arr.size > 100:
                    candidates.append((k, arr))
            except Exception:
                pass

        if len(candidates) > 0:
            candidates = sorted(candidates, key=lambda x: x[1].size, reverse=True)
            print("Using fallback key:", candidates[0][0], "shape:", candidates[0][1].shape)
            return candidates[0][1]

    raise ValueError(f"Cannot parse emotion2vec output: {item.keys() if isinstance(item, dict) else type(item)}")


def extract_e2v_sequence(audio, uid, tmp_dir, cfg):
    tmp_path = tmp_dir / f"{uid}.wav"
    sf.write(tmp_path, audio, cfg["sample_rate"])

    res = e2v_model.generate(
        input=str(tmp_path),
        granularity="frame",
        extract_embedding=True
    )

    emb = parse_e2v_frame_output(res)
    emb = np.asarray(emb, dtype=np.float32)

    # Robust shape handling
    if emb.ndim == 1:
        # fallback: one utterance vector only
        emb = emb.reshape(1, -1)

    elif emb.ndim == 3:
        # e.g. [1, T, D]
        if emb.shape[0] == 1:
            emb = emb[0]
        else:
            emb = emb.reshape(-1, emb.shape[-1])

    elif emb.ndim > 3:
        emb = emb.reshape(-1, emb.shape[-1])

    emb = np.nan_to_num(emb, nan=0.0, posinf=0.0, neginf=0.0)

    expected_dim = cfg.get("expected_e2v_dim", None)
    if expected_dim is not None and emb.shape[-1] != expected_dim:
        raise ValueError(
            f"Unexpected emotion2vec frame embedding dimension: got {emb.shape[-1]}, "
            f"expected {expected_dim}. "
            "This may indicate that the parser extracted the wrong output field."
        )

    return emb.astype(np.float32)

In [9]:
def resample_sequence_to_target(seq, target_frames):
    """
    Linear interpolation along time.
    Input : [T, D]
    Output: [target_frames, D]
    """
    seq = np.asarray(seq, dtype=np.float32)

    if seq.ndim != 2:
        raise ValueError(f"Expected 2D seq, got {seq.shape}")

    T, D = seq.shape

    if T == target_frames:
        return seq

    if T <= 1:
        return np.repeat(seq, target_frames, axis=0)

    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, target_frames)

    out = np.zeros((target_frames, D), dtype=np.float32)

    for d in range(D):
        out[:, d] = np.interp(new_idx, old_idx, seq[:, d])

    return out.astype(np.float32)


def make_valid_mask(original_frames, target_frames):
    """
    Karena audio sudah crop/pad ke durasi tetap 4 detik,
    kita gunakan mask semua valid setelah resampling.
    """
    return np.ones((target_frames,), dtype=np.float32)

In [10]:
def extract_split_sequence(df_split, dataset_name, split_name, cfg):
    X_hc = []
    X_e2v = []
    masks = []
    rows = []
    errors = []

    split_tmp_dir = TMP_AUDIO_DIR / dataset_name / split_name
    split_tmp_dir.mkdir(parents=True, exist_ok=True)

    target_frames = cfg["target_frames"]

    for idx, row in tqdm(
        df_split.iterrows(),
        total=len(df_split),
        desc=f"{dataset_name} {split_name}"
    ):
        try:
            audio = load_audio_fixed(
                row["filepath"],
                sr=cfg["sample_rate"],
                duration=cfg["duration"]
            )

            uid = str(row["uid"]).replace("/", "_").replace(" ", "_")

            hc_seq = extract_handcrafted_sequence(audio, cfg)
            e2v_seq = extract_e2v_sequence(audio, uid, split_tmp_dir, cfg)

            hc_seq = resample_sequence_to_target(hc_seq, target_frames)
            e2v_seq = resample_sequence_to_target(e2v_seq, target_frames)

            mask = make_valid_mask(target_frames, target_frames)

            X_hc.append(hc_seq)
            X_e2v.append(e2v_seq)
            masks.append(mask)

            rows.append(row.to_dict())

        except Exception as e:
            errors.append({
                "idx": idx,
                "uid": row.get("uid", ""),
                "filepath": row.get("filepath", ""),
                "error": str(e)
            })

    if len(X_hc) == 0:
        raise RuntimeError(f"No valid sequence for {dataset_name} {split_name}")

    X_hc = np.asarray(X_hc, dtype=np.float32)
    X_e2v = np.asarray(X_e2v, dtype=np.float32)
    masks = np.asarray(masks, dtype=np.float32)

    meta = pd.DataFrame(rows)
    y = meta["emotion"].map(LABEL_TO_ID).values.astype(np.int64)
    err = pd.DataFrame(errors)

    return X_e2v, X_hc, masks, y, meta, err

In [11]:
def fit_transform_sequence_scaler(X_train, X_val, X_test):
    """
    X shape: [N, T, D]
    Fit scaler pada train frames only.
    """
    N, T, D = X_train.shape

    scaler = StandardScaler()

    train_2d = X_train.reshape(-1, D)
    scaler.fit(train_2d)

    X_train_scaled = scaler.transform(X_train.reshape(-1, D)).reshape(X_train.shape).astype(np.float32)
    X_val_scaled = scaler.transform(X_val.reshape(-1, D)).reshape(X_val.shape).astype(np.float32)
    X_test_scaled = scaler.transform(X_test.reshape(-1, D)).reshape(X_test.shape).astype(np.float32)

    return X_train_scaled, X_val_scaled, X_test_scaled, scaler

In [12]:
def process_one_dataset_sequence(dataset_name, csv_path, cfg):
    print("=" * 90)
    print(f"Processing sequence features: {dataset_name.upper()}")
    print("=" * 90)

    out_ds = OUT_DIR / dataset_name
    out_ds.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(csv_path)
    df = df[df["emotion"].isin(LABELS)].copy()
    df["label"] = df["emotion"].map(LABEL_TO_ID).astype(int)

    print("Rows:", len(df))
    print("Speakers:", df["speaker"].nunique())
    display(df.groupby(["split", "emotion"]).size().unstack(fill_value=0))

    split_outputs = {}
    all_errors = []

    for split_name in ["train", "val", "test"]:
        df_split = df[df["split"] == split_name].reset_index(drop=True)

        X_e2v, X_hc, mask, y, meta, err = extract_split_sequence(
            df_split=df_split,
            dataset_name=dataset_name,
            split_name=split_name,
            cfg=cfg
        )

        split_outputs[split_name] = {
            "X_e2v": X_e2v,
            "X_hc": X_hc,
            "mask": mask,
            "y": y,
            "meta": meta,
            "err": err
        }

        print(
            split_name,
            "E2V:", X_e2v.shape,
            "HC:", X_hc.shape,
            "mask:", mask.shape,
            "y:", y.shape,
            "errors:", len(err)
        )

        if len(err) > 0:
            err["split"] = split_name
            all_errors.append(err)

    # Scaling train-only
    X_e2v_train, X_e2v_val, X_e2v_test, scaler_e2v = fit_transform_sequence_scaler(
        split_outputs["train"]["X_e2v"],
        split_outputs["val"]["X_e2v"],
        split_outputs["test"]["X_e2v"]
    )

    X_hc_train, X_hc_val, X_hc_test, scaler_hc = fit_transform_sequence_scaler(
        split_outputs["train"]["X_hc"],
        split_outputs["val"]["X_hc"],
        split_outputs["test"]["X_hc"]
    )

    # Save arrays
    save_map = {
        "train": (X_e2v_train, X_hc_train),
        "val": (X_e2v_val, X_hc_val),
        "test": (X_e2v_test, X_hc_test),
    }

    for split_name in ["train", "val", "test"]:
        X_e2v_scaled, X_hc_scaled = save_map[split_name]

        np.save(out_ds / f"X_e2v_seq_{split_name}.npy", X_e2v_scaled)
        np.save(out_ds / f"X_hc_seq_{split_name}.npy", X_hc_scaled)
        np.save(out_ds / f"mask_{split_name}.npy", split_outputs[split_name]["mask"])
        np.save(out_ds / f"y_{split_name}.npy", split_outputs[split_name]["y"])

        split_outputs[split_name]["meta"].to_csv(
            out_ds / f"meta_{split_name}.csv",
            index=False
        )

    # Save scalers
    import pickle

    with open(out_ds / "scaler_e2v_seq.pkl", "wb") as f:
        pickle.dump(scaler_e2v, f)

    with open(out_ds / "scaler_hc_seq.pkl", "wb") as f:
        pickle.dump(scaler_hc, f)

    if len(all_errors) > 0:
        errors_df = pd.concat(all_errors, ignore_index=True)
    else:
        errors_df = pd.DataFrame(columns=["idx", "uid", "filepath", "error", "split"])

    errors_df.to_csv(out_ds / "errors.csv", index=False)

    feature_config = {
        **cfg,
        "dataset": dataset_name,
        "e2v_dim": int(X_e2v_train.shape[-1]),
        "hc_dim": int(X_hc_train.shape[-1]),
        "target_frames": int(cfg["target_frames"]),
        "label_to_id": LABEL_TO_ID,
        "scaling": "StandardScaler fit on train frames only, separately for e2v and handcrafted",
        "note": "sequence-level features for temporal cross-attention; no augmentation",
    }

    with open(out_ds / "feature_config.json", "w") as f:
        json.dump(feature_config, f, indent=2)

    print("Saved:", out_ds)
    print("Errors:", len(errors_df))

    return {
        "dataset": dataset_name,
        "e2v_shape_train": X_e2v_train.shape,
        "hc_shape_train": X_hc_train.shape,
        "n_errors": len(errors_df),
    }

In [13]:
summary_rows = []

for dataset_name in DATASETS:
    result = process_one_dataset_sequence(
        dataset_name=dataset_name,
        csv_path=CSV_FILES[dataset_name],
        cfg=CONFIG
    )
    summary_rows.append(result)

summary_seq = pd.DataFrame(summary_rows)
summary_seq.to_csv(OUT_DIR / "sequence_extraction_summary.csv", index=False)

display(summary_seq)
print("Saved:", OUT_DIR / "sequence_extraction_summary.csv")

Processing sequence features: EMODB
Rows: 718
Speakers: 10


emotion,angry,disgust,fear,happy,neutral,sad
split,,,,,,
test,29,22,25,24,22,27
train,98,73,85,82,74,86
val,13,11,13,12,10,12


100%|██████████| 1/1 [00:01<00:00,  1.29s/it]
{'load_data': '0.373', 'extract_feat': 0.0, 'forward': '1.293', 'batch_size': '1', 'rtf': '0.323'}, : 100%|██████████| 1/1 [00:01<00:00,  1.29s/it]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.055', 'extract_feat': 0.0, 'forward': '0.070', 'batch_size': '1', 'rtf': '0.018'}, : 100%|██████████| 1/1 [00:00<00:00, 14.15it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.024', 'extract_feat': 0.0, 'forward': '0.040', 'batch_size': '1', 'rtf': '0.010'}, : 100%|██████████| 1/1 [00:00<00:00, 24.44it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.029', 'extract_feat': 0.0, 'forward': '0.046', 'batch_size': '1', 'rtf': '0.012'}, : 100%|██████████| 1/1 [00:00<00:00, 21.37it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.024', 'extract_feat': 0.0, 'forward': '0.041', 'batch_size': '1', 'rtf': '0.010'}, : 100%|██████████| 1/1 [00:00<00:00, 24.04it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.035', '

train E2V: (498, 200, 768) HC: (498, 200, 43) mask: (498, 200) y: (498,) errors: 0


  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.064', 'extract_feat': 0.0, 'forward': '0.080', 'batch_size': '1', 'rtf': '0.020'}, : 100%|██████████| 1/1 [00:00<00:00, 12.41it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.042', 'extract_feat': 0.0, 'forward': '0.062', 'batch_size': '1', 'rtf': '0.015'}, : 100%|██████████| 1/1 [00:00<00:00, 15.95it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.043', 'extract_feat': 0.0, 'forward': '0.063', 'batch_size': '1', 'rtf': '0.016'}, : 100%|██████████| 1/1 [00:00<00:00, 15.60it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.030', 'extract_feat': 0.0, 'forward': '0.048', 'batch_size': '1', 'rtf': '0.012'}, : 100%|██████████| 1/1 [00:00<00:00, 19.29it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.025', 'extract_feat': 0.0, 'forward': '0.042', 'batch_size': '1', 'rtf': '0.011'}, : 100%|██████████| 1/1 [00:00<00:00, 23.18it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.030', 'extract_

...

val E2V: (186, 200, 768) HC: (186, 200, 43) mask: (186, 200) y: (186,) errors: 0


  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.065', 'extract_feat': 0.0, 'forward': '0.084', 'batch_size': '1', 'rtf': '0.021'}, : 100%|██████████| 1/1 [00:00<00:00, 11.77it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.066', 'extract_feat': 0.0, 'forward': '0.086', 'batch_size': '1', 'rtf': '0.022'}, : 100%|██████████| 1/1 [00:00<00:00, 11.47it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.075', 'extract_feat': 0.0, 'forward': '0.096', 'batch_size': '1', 'rtf': '0.024'}, : 100%|██████████| 1/1 [00:00<00:00, 10.30it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.044', 'extract_feat': 0.0, 'forward': '0.061', 'batch_size': '1', 'rtf': '0.015'}, : 100%|██████████| 1/1 [00:00<00:00, 16.30it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.046', 'extract_feat': 0.0, 'forward': '0.068', 'batch_size': '1', 'rtf': '0.017'}, : 100%|██████████| 1/1 [00:00<00:00, 14.35it/s]
  0%|          | 0/1 [00:00<?, ?it/s]
{'load_data': '0.066', 'extract_

test E2V: (139, 200, 768) HC: (139, 200, 43) mask: (139, 200) y: (139,) errors: 0
Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base/resd
Errors: 0


,dataset,e2v_shape_train,hc_shape_train,n_errors
0,emodb,"(498, 200, 768)","(498, 200, 43)",0
1,ravdess,"(704, 200, 768)","(704, 200, 43)",0
2,resd,"(873, 200, 768)","(873, 200, 43)",0


Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base/sequence_extraction_summary.csv


In [14]:
for dataset_name in DATASETS:
    ds_dir = OUT_DIR / dataset_name

    print("=" * 80)
    print(dataset_name.upper())

    for split in ["train", "val", "test"]:
        X_e2v = np.load(ds_dir / f"X_e2v_seq_{split}.npy")
        X_hc = np.load(ds_dir / f"X_hc_seq_{split}.npy")
        mask = np.load(ds_dir / f"mask_{split}.npy")
        y = np.load(ds_dir / f"y_{split}.npy")

        print(
            split,
            "E2V:", X_e2v.shape,
            "HC:", X_hc.shape,
            "mask:", mask.shape,
            "y:", y.shape,
            "NaN E2V:", np.isnan(X_e2v).any(),
            "NaN HC:", np.isnan(X_hc).any()
        )

EMODB
train E2V: (498, 200, 768) HC: (498, 200, 43) mask: (498, 200) y: (498,) NaN E2V: False NaN HC: False
val E2V: (71, 200, 768) HC: (71, 200, 43) mask: (71, 200) y: (71,) NaN E2V: False NaN HC: False
test E2V: (149, 200, 768) HC: (149, 200, 43) mask: (149, 200) y: (149,) NaN E2V: False NaN HC: False
RAVDESS
train E2V: (704, 200, 768) HC: (704, 200, 43) mask: (704, 200) y: (704,) NaN E2V: False NaN HC: False
val E2V: (176, 200, 768) HC: (176, 200, 43) mask: (176, 200) y: (176,) NaN E2V: False NaN HC: False
test E2V: (176, 200, 768) HC: (176, 200, 43) mask: (176, 200) y: (176,) NaN E2V: False NaN HC: False
RESD
train E2V: (873, 200, 768) HC: (873, 200, 43) mask: (873, 200) y: (873,) NaN E2V: False NaN HC: False
val E2V: (186, 200, 768) HC: (186, 200, 43) mask: (186, 200) y: (186,) NaN E2V: False NaN HC: False
test E2V: (139, 200, 768) HC: (139, 200, 43) mask: (139, 200) y: (139,) NaN E2V: False NaN HC: False


In [15]:
# ============================================================
# SHA256 manifest for generated sequence feature files
# ============================================================
sequence_feature_manifest_rows = []

for path in sorted(OUT_DIR.rglob("*")):
    if path.is_file() and path.suffix.lower() in [".npy", ".csv", ".json", ".pkl"]:
        if path.name == "generated_sequence_feature_file_manifest_sha256.csv":
            continue

        sequence_feature_manifest_rows.append({
            "relative_path": str(path.relative_to(OUT_DIR)),
            "absolute_path": str(path),
            "size_bytes": path.stat().st_size,
            "sha256": sha256_of_file(path),
        })

sequence_feature_manifest = pd.DataFrame(sequence_feature_manifest_rows)
sequence_feature_manifest.to_csv(
    OUT_DIR / "generated_sequence_feature_file_manifest_sha256.csv",
    index=False
)

display(sequence_feature_manifest.head())
print("Saved:", OUT_DIR / "generated_sequence_feature_file_manifest_sha256.csv")
print("Number of files hashed:", len(sequence_feature_manifest))

,relative_path,absolute_path,size_bytes,sha256
0,emodb/X_e2v_seq_test.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,91545728,e4f2d821e110ecbea46d670ca7ddfe653b94cb00f512fd...
1,emodb/X_e2v_seq_train.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,305971328,008010ade43f83a69de18d086514537322f810a341a2f0...
2,emodb/X_e2v_seq_val.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,43622528,17e56db66e550d1d9c971c418a4edb7c84ec925d7c95e3...
3,emodb/X_hc_seq_test.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,5125728,666abf5ed8b746f4ff6bbc654249e4a96daa9d9dc9cdbc...
4,emodb/X_hc_seq_train.npy,/content/drive/MyDrive/New Jurnal Cross/proces...,17131328,c25685dd2fc7ba9dfb2358d510c6790c3f96dc6955e97b...


Saved: /content/drive/MyDrive/New Jurnal Cross/processed_intra_sequence_features_plus_base/generated_sequence_feature_file_manifest_sha256.csv
Number of files hashed: 59
